In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)


project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


In [2]:
from enviroment_bj import BlackjackTextGame, BlackjackConfig, ObservationConfig

game = BlackjackTextGame(
    config=BlackjackConfig(
        n_decks=1,
        shoe_penetration=1.0,
        observation=ObservationConfig.for_profile("minimal_basic_strategy"),
    ),
    seed=42,
)

game.new_round()


Round 1
Place your bet.
Actions: bet_1x, bet_2x, bet_3x, bet_4x

In [ ]:
game.bet_2x()

Round 1
Dealer: J ? (10)
-> You[0]: 10 K (20)
Actions: stand, hit, double, split, surrender

In [4]:
game.stand()


Round 1
Dealer: J 4 9 (23)
   You[0]: 10 K (20) | win, +2.00
Round result: +2.00
Use game.new_round() to play again.

In [6]:
game.new_round()


Round 2
Place your bet.
Actions: bet_1x, bet_2x, bet_3x, bet_4x

In [7]:

game.bet_4x()

Round 2
Dealer: 4 7 (11)
-> You[0]: K A (21) | soft, blackjack, +6.00
Round result: +6.00
Use game.new_round() to play again.

In [6]:
from enviroment_bj import BlackjackJSONWrapper, BlackjackConfig, ObservationConfig

wrapper = BlackjackJSONWrapper(
    config=BlackjackConfig(
        n_decks=1,
        shoe_penetration=1.0,
        observation=ObservationConfig.for_profile("minimal_basic_strategy"),
    ),
    seed=11,
)

state = wrapper.reset()

print("done:", state["done"])
print("decision_phase:", state["observation"]["decision_phase"])
print("legal_actions:", state["legal_actions"])
print("current_bet:", state["observation"].get("current_bet"))
print("dealer_upcard:", state["observation"].get("dealer_upcard"))


done: False
decision_phase: betting
legal_actions: ['bet_1x', 'bet_2x', 'bet_3x', 'bet_4x']
current_bet: None
dealer_upcard: None


In [7]:
state = wrapper.step("bet_2x")
public = state["info"]["public_state"]

print("done:", state["done"])
print("decision_phase:", state["observation"]["decision_phase"])
print("current_bet:", public["current_bet"])
print("dealer_upcard:", state["observation"]["dealer_upcard"])
print("player_cards:", public["current_hand"]["cards"])
print("player_total:", public["current_hand"]["total"])
print("legal_actions:", state["legal_actions"])
print("action_mask:", state["action_mask"])


done: False
decision_phase: playing
current_bet: 2.0
dealer_upcard: 5
player_cards: ['6', '5']
player_total: 11
legal_actions: ['stand', 'hit', 'double', 'surrender']
action_mask: [0, 0, 0, 0, 1, 1, 1, 0, 1, 0]


In [8]:
state = wrapper.reset()
print("fase inicial:", state["observation"]["decision_phase"])

state = wrapper.step("bet_1x")
public = state["info"]["public_state"]

print("cartas jugador:", public["current_hand"]["cards"])
print("total jugador:", public["current_hand"]["total"])
print("dealer upcard:", state["observation"]["dealer_upcard"])

while not state["done"]:
    public = state["info"]["public_state"]
    total = public["current_hand"]["total"]
    legal = state["legal_actions"]

    if "double" in legal and total in (10, 11):
        action = "double"
    elif "hit" in legal and total < 17:
        action = "hit"
    else:
        action = "stand"

    print("accion:", action)
    state = wrapper.step(action)

print("done:", state["done"])
print("reward:", state["reward"])
print("dealer final:", state["info"]["public_state"]["dealer"])
print("player_hands:", state["info"]["public_state"]["player_hands"])


fase inicial: betting
cartas jugador: ['K', '10']
total jugador: 20
dealer upcard: 6
accion: stand
done: True
reward: 1.0
dealer final: {'upcard': '6', 'cards': ['6', 'J', '9'], 'hole_card_hidden': False, 'visible_total': 25, 'visible_is_soft': False, 'peek_checked': False, 'has_blackjack': False, 'total': 25, 'is_soft': False}
player_hands: [{'index': 0, 'cards': ['K', '10'], 'total': 20, 'is_soft': False, 'is_blackjack': False, 'is_bust': False, 'bet': 1.0, 'doubled': False, 'from_split': False, 'split_aces': False, 'closed': True, 'surrendered': False, 'action_count': 1, 'close_reason': 'stand', 'settlement': 'win', 'reward': 1.0}]


In [9]:
from enviroment_bj import BlackjackJSONWrapper, BlackjackConfig, ObservationConfig

wrapper = BlackjackJSONWrapper(
    config=BlackjackConfig(
        n_decks=1,
        shoe_penetration=1.0,
        observation=ObservationConfig.for_profile("minimal_basic_strategy"),
    ),
    seed=11,
)

wrapper.environment.load_shoe(
    ["10", "6", "7", "10", "10", "9", "5", "2", "10", "K", "8"],
    total_cards=11,
)

state = wrapper.reset()
print("reset legal_actions:", state["legal_actions"])
print("phase:", state["observation"]["decision_phase"])

state = wrapper.step("bet_1x")
public = state["info"]["public_state"]

print("player:", public["current_hand"]["cards"], public["current_hand"]["total"])
print("bet:", public["current_bet"])
print("dealer:", state["observation"]["dealer_upcard"])
print("legal_actions:", state["legal_actions"])

state = wrapper.step("stand")
print("done:", state["done"])
print("reward:", state["reward"])
print("dealer final:", state["info"]["public_state"]["dealer"])
print("settlement:", state["info"]["public_state"]["player_hands"][0]["settlement"])


reset legal_actions: ['bet_1x', 'bet_2x', 'bet_3x', 'bet_4x']
phase: betting
player: ['10', '7'] 17
bet: 1.0
dealer: 6
legal_actions: ['stand', 'hit', 'double', 'surrender']
done: True
reward: 1.0
dealer final: {'upcard': '6', 'cards': ['6', '10', '10'], 'hole_card_hidden': False, 'visible_total': 26, 'visible_is_soft': False, 'peek_checked': False, 'has_blackjack': False, 'total': 26, 'is_soft': False}
settlement: win


In [10]:
wrapper.environment.load_shoe(["9", "A", "7", "K"], total_cards=4)

state = wrapper.reset()
print("betting actions:", state["legal_actions"])

state = wrapper.step("bet_4x")
print("legal after bet:", state["legal_actions"])
print("current_bet:", state["info"]["public_state"]["current_bet"])

state = wrapper.step("insurance")
print("done:", state["done"])
print("reward:", state["reward"])
print("insurance:", state["info"]["public_state"]["insurance"])
print("dealer:", state["info"]["public_state"]["dealer"])


betting actions: ['bet_1x', 'bet_2x', 'bet_3x', 'bet_4x']
legal after bet: ['stand', 'hit', 'double', 'surrender', 'insurance']
current_bet: 4.0
done: True
reward: 0.0
insurance: {'offer_active': False, 'bet': 2.0, 'reward': 4.0}
dealer: {'upcard': 'A', 'cards': ['A', 'K'], 'hole_card_hidden': False, 'visible_total': 21, 'visible_is_soft': True, 'peek_checked': True, 'has_blackjack': True, 'total': 21, 'is_soft': True}


In [11]:
wrapper.environment.load_shoe(["8", "6", "8", "10", "3", "K", "2", "10"], total_cards=8)

state = wrapper.reset()
state = wrapper.step("bet_1x")

print("player:", state["info"]["public_state"]["current_hand"]["cards"])
print("legal:", state["legal_actions"])

state = wrapper.step("split")
print("after split hands:", state["info"]["public_state"]["player_hands"])
print("current_hand_index:", state["info"]["public_state"]["current_hand_index"])
print("legal:", state["legal_actions"])

state = wrapper.step("double")
print("after double current_hand_index:", state["info"]["public_state"]["current_hand_index"])
print("current_hand:", state["info"]["public_state"]["current_hand"])

state = wrapper.step("stand")
print("done:", state["done"])
print("reward:", state["reward"])
print("dealer:", state["info"]["public_state"]["dealer"])
print("hands:", state["info"]["public_state"]["player_hands"])


player: ['8', '8']
legal: ['stand', 'hit', 'double', 'split', 'surrender']
after split hands: [{'index': 0, 'cards': ['8', '3'], 'total': 11, 'is_soft': False, 'is_blackjack': False, 'is_bust': False, 'bet': 1.0, 'doubled': False, 'from_split': True, 'split_aces': False, 'closed': False, 'surrendered': False, 'action_count': 0, 'close_reason': None, 'settlement': None, 'reward': 0.0}, {'index': 1, 'cards': ['8', 'K'], 'total': 18, 'is_soft': False, 'is_blackjack': False, 'is_bust': False, 'bet': 1.0, 'doubled': False, 'from_split': True, 'split_aces': False, 'closed': False, 'surrendered': False, 'action_count': 0, 'close_reason': None, 'settlement': None, 'reward': 0.0}]
current_hand_index: 0
legal: ['stand', 'hit', 'double']
after double current_hand_index: 1
current_hand: {'index': 1, 'cards': ['8', 'K'], 'total': 18, 'is_soft': False, 'is_blackjack': False, 'is_bust': False, 'bet': 1.0, 'doubled': False, 'from_split': True, 'split_aces': False, 'closed': False, 'surrendered': False

In [12]:
def resumen(state):
    public = state["info"]["public_state"]
    current_hand = public["current_hand"]

    return {
        "done": state["done"],
        "reward": state["reward"],
        "phase": state["observation"].get("decision_phase"),
        "legal_actions": state["legal_actions"],
        "current_bet": public["current_bet"],
        "player_cards": None if current_hand is None else current_hand["cards"],
        "player_total": None if current_hand is None else current_hand["total"],
        "dealer_upcard": state["observation"].get("dealer_upcard"),
        "dealer_public": public["dealer"],
    }

resumen(state)


{'done': True,
 'reward': 3.0,
 'phase': 'playing',
 'legal_actions': [],
 'current_bet': None,
 'player_cards': None,
 'player_total': None,
 'dealer_upcard': '6',
 'dealer_public': {'upcard': '6',
  'cards': ['6', '10', '10'],
  'hole_card_hidden': False,
  'visible_total': 26,
  'visible_is_soft': False,
  'peek_checked': False,
  'has_blackjack': False,
  'total': 26,
  'is_soft': False}}